In [1]:
import pandas as pd
import numpy as np
import math

In [2]:
df_shot=pd.read_csv('shots.csv')
df_shot=df_shot[df_shot['shot_type']=='Open Play']
df_shot=df_shot[df_shot['shot_body_part']=='Head']

In [3]:
def calculateDistance(x,y):
    x_distance=120-x
    y_distance=0
    if (y<36):
        y_distance = 36-y
    elif (y>44):
        y_distance = y-44
    return np.sqrt(y_distance**2+x_distance**2)

def calculateAngle(x,y):
    g0 = [120, 44]
    p = [x, y]
    g1 = [120, 36]

    v0 = np.array(g0) - np.array(p)
    v1 = np.array(g1) - np.array(p)

    angle = math.atan2(np.linalg.det([v0,v1]),np.dot(v0,v1))
    return(abs(np.degrees(angle)))

def calculateDistanceShooterGk(x1,y1,x2,y2):
    return np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

In [4]:
df_shot['angle'] = df_shot.apply(lambda row:calculateAngle(row['x'], row['y']), axis=1)
df_shot['distance'] = df_shot.apply(lambda row:calculateDistance(row['x'], row['y']), axis=1)
df_shot['1on1'] = df_shot.apply(lambda row:1 if row['shot_one_on_one']==True else 0, axis=1)
df_shot['underPressure'] = df_shot.apply(lambda row:1 if row['under_pressure']==True else 0, axis=1)

df_shot['DistanceShooterGk'] = df_shot.apply(
    lambda row: calculateDistanceShooterGk(row['x'], row['y'], row['x_gk'], row['y_gk']) 
    if not row['y_gk']!=np.nan else np.nan, 
    axis=1
)
df_shot['DistanceGk'] = df_shot.apply(
    lambda row: calculateDistance(row['x_gk'], row['y_gk'])
    if not row['y_gk']!=np.nan else np.nan,
     axis=1
)
df_shot['minus'] = df_shot.apply(
    lambda row: row['x']-row['x_gk'], axis=1
)

In [4]:
df_shot.to_csv('shots_openplay_head.csv', index=False)